# MultiCloudTenant01 — Scan & Replication Plan

**Tenant:** MultiCloudTenant01  
**Subscription:** Purview_Canary_M365_Multicloud

> Tenant ID and subscription ID are loaded from `.env` — see `.env.example` for required variables.

This notebook covers three phases:

1. **Scan** — Discover all Azure resources in the tenant and populate the Neo4j graph.
2. **Replication Plan** — Analyse the discovered graph, detect architectural patterns, and generate a replication plan using spectral graph analysis.
3. **Deployment Complexity Scoring** — Score each pattern and selected instance by how long and how difficult it would be to deploy, grounded in Microsoft Learn deployment data.

---

## Setup

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import numpy as np

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv())

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

# ── Tenant identity (loaded from .env) ────────────────────────────────────────
TENANT_ALIAS      = os.getenv("MULTICLOUD_TENANT_ALIAS",      "MultiCloudTenant01")
TENANT_ID         = os.getenv("MULTICLOUD_TENANT_ID")
SUBSCRIPTION_ID   = os.getenv("MULTICLOUD_SUBSCRIPTION_ID")
SUBSCRIPTION_NAME = os.getenv("MULTICLOUD_SUBSCRIPTION_NAME", "")

if not TENANT_ID:
    raise EnvironmentError("MULTICLOUD_TENANT_ID is not set — add it to your .env file")
if not SUBSCRIPTION_ID:
    raise EnvironmentError("MULTICLOUD_SUBSCRIPTION_ID is not set — add it to your .env file")

# ── Neo4j connection ───────────────────────────────────────────────────────────
NEO4J_URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER",     "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

if not NEO4J_PASSWORD:
    raise EnvironmentError("NEO4J_PASSWORD is not set — add it to your .env file")

print(f"Tenant  : {TENANT_ALIAS} ({TENANT_ID})")
print(f"Sub     : {SUBSCRIPTION_NAME} ({SUBSCRIPTION_ID})")
print(f"Neo4j   : {NEO4J_URI}")
print("✅ Setup complete")

---
# Phase 1: Scan
---

Discover all resources in **MultiCloudTenant01** and populate Neo4j.  
The scan uses `cli.py scan` which auto-starts the Neo4j container if needed.

> Set `SKIP_SCAN = True` if the graph is already populated.

In [ ]:
SKIP_SCAN = True  # Set True to skip if graph already exists

if SKIP_SCAN:
    print("⏭️  Scan skipped — using existing graph data.")
else:
    CLI = REPO_ROOT / "scripts" / "cli.py"
    cmd = [
        sys.executable, str(CLI),
        "scan",
        "--tenant-id", TENANT_ID,
        "--no-dashboard",
    ]
    print(f"🔍 Starting scan for {TENANT_ALIAS} …")
    print(f"   Command: {' '.join(cmd)}\n")
    result = subprocess.run(cmd, cwd=str(REPO_ROOT), capture_output=False, text=True)
    if result.returncode == 0:
        print("\n✅ Scan completed successfully.")
    else:
        print(f"\n❌ Scan exited with code {result.returncode}")
        raise RuntimeError("Scan failed — check output above for details.")

### Verify Graph Contents

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    counts = session.run(
        "MATCH (n) RETURN labels(n)[0] AS label, count(n) AS count ORDER BY count DESC"
    ).data()
    rel_counts = session.run(
        "MATCH ()-[r]->() RETURN type(r) AS rel, count(r) AS count ORDER BY count DESC"
    ).data()
driver.close()

print(f"📊 Graph contents for {TENANT_ALIAS}:\n")
print(f"  {'Node Type':<30} {'Count'}")
print(f"  {'-'*40}")
for row in counts:
    print(f"  {str(row['label']):<30} {row['count']}")
total_nodes = sum(r["count"] for r in counts)
print(f"  {'─'*40}")
print(f"  {'TOTAL':<30} {total_nodes}")
print(f"\n  {'Relationship Type':<30} {'Count'}")
print(f"  {'-'*40}")
for row in rel_counts:
    print(f"  {str(row['rel']):<30} {row['count']}")

---
# Phase 2: Replication Plan
---

Analyse the **MultiCloudTenant01** graph, detect architectural patterns, and generate a replication plan.

## Analyse Source Tenant

In [ ]:
import importlib
import src.architecture_based_replicator as _abr_mod
importlib.reload(_abr_mod)
from src.architecture_based_replicator import ArchitecturePatternReplicator
from src.replicator.modules.graph_structure_analyzer import GraphStructureAnalyzer

replicator = ArchitecturePatternReplicator(
    neo4j_uri=NEO4J_URI,
    neo4j_user=NEO4J_USER,
    neo4j_password=NEO4J_PASSWORD,
)

print(f"🔍 Analysing {TENANT_ALIAS} with configuration coherence …")
analysis = replicator.analyze_source_tenant(
    use_configuration_coherence=True,
    coherence_threshold=0.5,
)

print(f"\n📊 Source Tenant: {TENANT_ALIAS}")
print(f"   Resource Types          : {analysis['resource_types']}")
print(f"   Pattern Graph Edges     : {analysis['pattern_graph_edges']}")
print(f"   Detected Patterns       : {analysis['detected_patterns']}")
print(f"   Total Pattern Instances : {analysis.get('total_pattern_resources', 0)}")

print("\n📐 Detected Architectural Patterns:")
for name, info in replicator.detected_patterns.items():
    instances = replicator.pattern_resources.get(name, [])
    total_res = sum(len(i) for i in instances)
    print(f"  {name}:")
    print(f"    Instances: {len(instances)}, Resources: {total_res}")

## Filter Instance Pool by Deployment Complexity

Before generating the plan, every candidate instance is scored with `score_instance_deployment` and those whose **combined score exceeds `MAX_DEPLOYMENT_SCORE`** are removed.

This writes the filtered pool back to `replicator.pattern_resources` so that `generate_replication_plan()` draws only from low-complexity instances.

| Score range | Band | Meaning |
|---|---|---|
| 1.0 – 3.0 | Low | Fast and simple — minutes, minimal config |
| 3.0 – 5.5 | Medium | Moderate time and config |
| 5.5 – 7.5 | High | Slow or complex (e.g. AKS, private endpoints) |
| 7.5 – 10 | Critical | Very slow or expert-level (VPN Gateway, SQL MI) |

In [ ]:
import importlib
import src.architectural_pattern_analyzer as _apa_mod
importlib.reload(_apa_mod)
from src.architectural_pattern_analyzer import ArchitecturalPatternAnalyzer

analyzer = ArchitecturalPatternAnalyzer(
    neo4j_uri=NEO4J_URI,
    neo4j_user=NEO4J_USER,
    neo4j_password=NEO4J_PASSWORD,
)

# Instances with combined_score > this are excluded from the plan.
# 4.0 keeps Low and easy-Medium instances only.
MAX_DEPLOYMENT_SCORE = 4.0

print(f"🔍 Scoring and filtering instances (threshold ≤ {MAX_DEPLOYMENT_SCORE}) …\n")

original_counts = {p: len(insts) for p, insts in replicator.pattern_resources.items()}
filtered_resources = {}

for pattern_name, instances in replicator.pattern_resources.items():
    kept = []
    for instance in instances:
        score = analyzer.score_instance_deployment(instance)
        if score["combined_score"] <= MAX_DEPLOYMENT_SCORE:
            kept.append(instance)
    filtered_resources[pattern_name] = kept

# Write filtered pool back so generate_replication_plan draws from it
replicator.pattern_resources = filtered_resources

print(f"{'Pattern':<32} {'Before':>8} {'After':>8} {'Removed':>9} {'% kept':>8}")
print("-" * 65)
total_before, total_after = 0, 0
for pname in original_counts:
    before  = original_counts[pname]
    after   = len(filtered_resources.get(pname, []))
    removed = before - after
    pct     = after / before * 100 if before else 0
    total_before += before
    total_after  += after
    print(f"  {pname:<30} {before:>8} {after:>8} {removed:>9} {pct:>7.1f}%")
print("-" * 65)
print(f"  {'TOTAL':<30} {total_before:>8} {total_after:>8} "
      f"{total_before - total_after:>9} {total_after/total_before*100:>7.1f}%")
print(f"\n✅ {total_after} of {total_before} instances kept (score ≤ {MAX_DEPLOYMENT_SCORE})")

In [ ]:
TARGET_INSTANCE_COUNT = 500

print(f"🔨 Generating replication plan for {TENANT_ALIAS} …")
print(f"   Drawing from low-complexity pool (score ≤ {MAX_DEPLOYMENT_SCORE})\n")
print("Parameters:")
print("  spectral_weight=0.4  (60% distribution, 40% structure)")
print("  use_architecture_distribution=True")
print("  use_configuration_coherence=True")
print("  use_spectral_guidance=True")
print("  include_colocated_orphaned_resources=True")
print("  max_config_samples=100")
print(f"  target_instance_count={TARGET_INSTANCE_COUNT}\n")

selected_pattern_instances, spectral_history, distribution_metadata = \
    replicator.generate_replication_plan(target_instance_count=TARGET_INSTANCE_COUNT)

total_resources = sum(len(inst) for _, inst in selected_pattern_instances)

# Verify all selected instances respect the complexity threshold
selected_scores = [
    analyzer.score_instance_deployment(inst)
    for _, inst in selected_pattern_instances
]
mean_score = sum(s["combined_score"] for s in selected_scores) / len(selected_scores) if selected_scores else 0
max_score  = max((s["combined_score"] for s in selected_scores), default=0)

print(f"✅ Replication plan generated for {TENANT_ALIAS}")
print(f"   Selected instances    : {len(selected_pattern_instances)}")
print(f"   Total resources       : {total_resources}")
print(f"   Mean deployment score : {mean_score:.2f} / 10")
print(f"   Max deployment score  : {max_score:.2f} / 10  (threshold was {MAX_DEPLOYMENT_SCORE})")

## Build Target Pattern Graph

In [ ]:
print("🔍 Building target pattern graph from selected instances …\n")
target_pattern_graph = replicator.target_builder.build_from_instances(selected_pattern_instances)
print(f"✅ Target pattern graph built for {TENANT_ALIAS}:")
print(f"   Nodes (resource types) : {target_pattern_graph.number_of_nodes()}")
print(f"   Edges (relationships)  : {target_pattern_graph.number_of_edges()}")
print(f"   Total resources        : {total_resources}")

## Graph Statistics Comparison

In [ ]:
source_graph = replicator.source_pattern_graph

source_nodes = source_graph.number_of_nodes()
target_nodes = target_pattern_graph.number_of_nodes()
source_edges = source_graph.number_of_edges()
target_edges = target_pattern_graph.number_of_edges()

spectral_distance = GraphStructureAnalyzer.compute_spectral_distance(
    source_graph, target_pattern_graph
)

print(f"📊 Pattern Graph Comparison — {TENANT_ALIAS}:\n")
print(f"{'Metric':<30} {'Source':<15} {'Target':<15} {'Coverage %'}")
print("-" * 75)
print(f"{'Nodes (resource types)':<30} {source_nodes:<15} {target_nodes:<15} {(target_nodes/source_nodes*100):.1f}%")
print(f"{'Edges (relationships)':<30} {source_edges:<15} {target_edges:<15} {(target_edges/source_edges*100):.1f}%")
print(f"\n📐 Spectral Distance: {spectral_distance:.4f}  (lower = better structural match)")

## Source Pattern Graph Visualization

Each **node** is a resource type present in the scanned tenant.  
Each **edge** represents a co-occurrence / dependency relationship detected during pattern analysis.  
Node size scales with degree (number of relationships). Node colour indicates the primary architectural pattern.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from matplotlib.patches import Patch, Polygon
from scipy.spatial import ConvexHull

TOP_N = 40

G = source_graph
pattern_matches = replicator.detected_patterns
resource_type_counts = replicator.source_resource_type_counts

# ── Reconstruct edge_counts from graph edge frequency attributes ───────────────
edge_counts: dict[tuple, int] = {}
for u, v, data in G.edges(data=True):
    key = (u, v)
    edge_counts[key] = edge_counts.get(key, 0) + data.get("frequency", 1)

# ── Select top N connected nodes by count ─────────────────────────────────────
# Restrict to degree > 0 nodes: these are the short-name nodes derived from
# relationships (e.g. "virtualMachines"). Orphan nodes added from
# all_resource_types use full ARM-type keys ("Microsoft.Compute/virtualMachines")
# which don't match pattern matched_resources and would crowd out pattern nodes.
connected_nodes = {n for n in G.nodes() if G.degree(n) > 0}
top_node_names = [
    name for name, _ in
    sorted(
        ((n, resource_type_counts.get(n, 0)) for n in connected_nodes),
        key=lambda x: x[1], reverse=True,
    )[:TOP_N]
]
G_filtered = G.subgraph(top_node_names).copy()
n_total = G.number_of_nodes()

print(f"Full graph      : {n_total} nodes, {G.number_of_edges()} edges")
print(f"Connected nodes : {len(connected_nodes)}")
print(f"Showing top {TOP_N}  : {G_filtered.number_of_nodes()} nodes, {G_filtered.number_of_edges()} edges")

# ── Layout ────────────────────────────────────────────────────────────────────
pos = nx.spring_layout(G_filtered, k=3, iterations=50, seed=42)

# ── Assign patterns to nodes ──────────────────────────────────────────────────
node_pattern_map: dict[str, list] = {}
node_colors: list = []

for node in G_filtered.nodes():
    node_patterns = [
        p for p, match in pattern_matches.items()
        if node in match["matched_resources"]
    ]
    node_pattern_map[node] = node_patterns
    if node_patterns:
        best = max(node_patterns, key=lambda p: pattern_matches[p]["completeness"])
        node_colors.append(list(pattern_matches.keys()).index(best))
    else:
        node_colors.append(-1)

# ── Node sizes: proportional to resource count ────────────────────────────────
node_sizes = [
    max(200, G_filtered.nodes[n].get("count", resource_type_counts.get(n, 1)) / 4)
    for n in G_filtered.nodes()
]

# ── Separate intra-pattern vs cross-pattern edges ─────────────────────────────
pattern_edges, cross_pattern_edges = [], []
pattern_edge_widths, cross_pattern_edge_widths = [], []
pattern_edge_colors = []

for u, v, data in G_filtered.edges(data=True):
    freq = edge_counts.get((u, v), 1)
    width = max(1, freq / 50)
    shared = set(node_pattern_map.get(u, [])) & set(node_pattern_map.get(v, []))
    if shared:
        pattern_edges.append((u, v))
        pattern_edge_widths.append(width * 2.5)
        pattern_edge_colors.append(list(pattern_matches.keys()).index(list(shared)[0]))
    else:
        cross_pattern_edges.append((u, v))
        cross_pattern_edge_widths.append(width * 0.4)

# ── Draw ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(28, 24))
cmap = plt.cm.tab10
pattern_legend = []

# Convex hull boundaries per pattern
hulls_drawn = 0
for pattern_idx, (pattern_name, match) in enumerate(pattern_matches.items()):
    p_nodes = [n for n in match["matched_resources"] if n in G_filtered.nodes()]
    if len(p_nodes) >= 3:
        try:
            pts = np.array([[pos[n][0], pos[n][1]] for n in p_nodes])
            center = pts.mean(axis=0)
            pts_exp = center + (pts - center) * 1.15
            hull = ConvexHull(pts_exp)
            hull_pts = pts_exp[hull.vertices]
            color = cmap(pattern_idx / len(pattern_matches))
            ax.add_patch(Polygon(
                hull_pts, facecolor=color, alpha=0.08,
                edgecolor=color, linewidth=3, linestyle="--", zorder=1,
            ))
            hulls_drawn += 1
            pattern_legend.append(Patch(
                facecolor=color, edgecolor=color,
                label=f"{pattern_name} ({match['completeness']:.0f}%)", alpha=0.5,
            ))
        except Exception:
            pass

# Cross-pattern edges (gray, thin)
if cross_pattern_edges:
    nx.draw_networkx_edges(
        G_filtered, pos, ax=ax,
        edgelist=cross_pattern_edges, width=cross_pattern_edge_widths,
        alpha=0.15, edge_color="gray",
        arrows=True, arrowsize=10, arrowstyle="->",
        connectionstyle="arc3,rad=0.05",
    )

# Intra-pattern edges (colored by pattern)
for idx, (u, v) in enumerate(pattern_edges):
    nx.draw_networkx_edges(
        G_filtered, pos, ax=ax,
        edgelist=[(u, v)], width=pattern_edge_widths[idx],
        alpha=0.6, edge_color=[cmap(pattern_edge_colors[idx] / len(pattern_matches))],
        arrows=True, arrowsize=15, arrowstyle="->",
        connectionstyle="arc3,rad=0.1",
    )

# Nodes
nx.draw_networkx_nodes(
    G_filtered, pos, ax=ax,
    node_size=node_sizes, node_color=node_colors,
    cmap=cmap, vmin=-1, vmax=len(pattern_matches) - 1,
    alpha=0.95, edgecolors="black", linewidths=2.5,
)

# Labels
nx.draw_networkx_labels(G_filtered, pos, ax=ax, font_size=10, font_weight="bold")

if pattern_legend:
    ax.legend(
        handles=pattern_legend, loc="upper left",
        fontsize=10, framealpha=0.95,
        title="Architectural Patterns", title_fontsize=12,
    )

ax.set_title(
    f"Source Pattern Graph — {TENANT_ALIAS}  (top {TOP_N} connected nodes by resource count)\n"
    "Dashed boundaries = Pattern groupings  ·  Thick colored edges = Intra-pattern  ·  "
    "Thin gray = Cross-pattern\n"
    "Node color = Pattern  ·  Node size ∝ resource count",
    fontsize=16, fontweight="bold", pad=25,
)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"\n📊 Visualization:")
print(f"   Resource types shown : {len(G_filtered.nodes())}")
print(f"   Intra-pattern edges  : {len(pattern_edges)} (colored)")
print(f"   Cross-pattern edges  : {len(cross_pattern_edges)} (gray)")
print(f"   Hulls drawn          : {hulls_drawn} / {len(pattern_matches)} patterns")

## Resource Types per Architecture Instance

Resource type nodes present in the **source pattern graph** for each detected architecture pattern.

In [ ]:
G_src = replicator.source_pattern_graph
graph_nodes = {n for n in G_src.nodes() if G_src.degree(n) > 0}

print(f"{'='*70}")
print(f"  SOURCE PATTERN GRAPH — Resource Types per Architecture Instance")
print(f"{'='*70}")

for pattern_name, pattern_info in replicator.detected_patterns.items():
    matched = pattern_info.get("matched_resources", [])
    # Intersect with nodes that actually appear in the graph
    in_graph = sorted(r for r in matched if r in graph_nodes)
    instances = replicator.pattern_resources.get(pattern_name, [])
    total_res = sum(len(inst) for inst in instances)

    print(f"\n  {pattern_name}")
    print(f"  {'─' * (len(pattern_name) + 2)}")
    print(f"  Instances : {len(instances)}   |   Resources across all instances : {total_res}")
    print(f"  Resource types in source graph ({len(in_graph)}):")
    if in_graph:
        for rt in in_graph:
            degree = G_src.degree(rt)
            print(f"    • {rt:<40s}  (degree {degree})")
    else:
        print("    (none matched in graph)")

print(f"\n{'='*70}")
print(f"  Total detected patterns : {len(replicator.detected_patterns)}")
print(f"  Graph nodes with edges  : {len(graph_nodes)}")
print(f"{'='*70}")

## Node Overlap Analysis

In [ ]:
source_nodes_set = set(source_graph.nodes())
target_nodes_set = set(target_pattern_graph.nodes())

common_nodes  = source_nodes_set & target_nodes_set
missing_nodes = source_nodes_set - target_nodes_set
extra_nodes   = target_nodes_set - source_nodes_set

print(f"📊 Node Overlap — {TENANT_ALIAS}:\n")
print(f"   Source unique       : {len(source_nodes_set)}")
print(f"   Target unique       : {len(target_nodes_set)}")
print(f"   Common              : {len(common_nodes)}")
print(f"   Missing from target : {len(missing_nodes)}")
print(f"   Extra in target     : {len(extra_nodes)}")

fig, ax = plt.subplots(figsize=(10, 5))
categories = ['Common\n(in both)', 'Source Only\n(missing from target)', 'Target Only\n(extra)']
values = [len(common_nodes), len(missing_nodes), len(extra_nodes)]
colors = ['#99ff99', '#ff9999', '#99ccff']
bars = ax.bar(categories, values, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            str(val), ha='center', va='bottom', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Resource Types', fontsize=12, fontweight='bold')
ax.set_title(f'Resource Type Overlap — {TENANT_ALIAS}', fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

coverage_pct = len(common_nodes) / len(source_nodes_set) * 100 if source_nodes_set else 0
print(f"\n📈 Coverage: {coverage_pct:.1f}% of source types included in target")

if missing_nodes:
    source_degrees = dict(source_graph.degree())
    missing_by_degree = sorted(
        [(n, source_degrees.get(n, 0)) for n in missing_nodes],
        key=lambda x: x[1], reverse=True
    )
    print(f"\n🔍 Top missing resource types:")
    for i, (node, degree) in enumerate(missing_by_degree[:10], 1):
        print(f"   {i}. {node} (degree: {degree})")

## Side-by-Side Graph Visualization

**Blue nodes** appear in both source and target. **Red nodes** are unique to that graph.
**Red edges** (source) = relationships present in source but absent from target.
**Green edges** (target) = relationships captured in the replication plan.

In [ ]:
from collections import Counter

TOP_N_SBS = 40

# ── Build subgraphs (top N by degree) ─────────────────────────────────────────
src_deg = dict(source_graph.degree())
top_src = [n for n, _ in sorted(src_deg.items(), key=lambda x: x[1], reverse=True)[:TOP_N_SBS]]
src_sub = source_graph.subgraph(top_src).copy()

tgt_deg = dict(target_pattern_graph.degree())
top_tgt = [n for n, _ in sorted(tgt_deg.items(), key=lambda x: x[1], reverse=True)[:TOP_N_SBS]]
tgt_sub = target_pattern_graph.subgraph(top_tgt).copy()

# ── Missing edges: in source but not in target ────────────────────────────────
src_eset = {(u, v, d.get("relationship","UNKNOWN")) for u,v,d in src_sub.edges(data=True)}
tgt_eset = {(u, v, d.get("relationship","UNKNOWN")) for u,v,d in tgt_sub.edges(data=True)}
missing_edges_sbs = [
    (u, v, rel) for u, v, rel in src_eset
    if u in target_nodes_set and v in target_nodes_set
    and (u, v, rel) not in tgt_eset
]

print(f"📊 Edge Analysis:")
print(f"   Source subgraph : {src_sub.number_of_edges()} edges")
print(f"   Target subgraph : {tgt_sub.number_of_edges()} edges")
print(f"   Missing edges   : {len(missing_edges_sbs)}")

# ── Drawing helper ────────────────────────────────────────────────────────────
def _draw_sbs(G, ax, title, highlight_missing=False):
    if G.number_of_nodes() == 0:
        ax.text(0.5, 0.5, "No nodes", ha="center", va="center",
                transform=ax.transAxes, fontsize=16)
        ax.set_title(title, fontsize=14, fontweight="bold", pad=20)
        ax.axis("off")
        return

    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    degs = dict(G.degree())
    node_sizes  = [degs[n] * 100 + 200 for n in G.nodes()]
    node_colors = ["#66b3ff" if n in common_nodes else "#ff9999" for n in G.nodes()]

    if highlight_missing and missing_edges_sbs:
        miss_set = {(u, v) for u, v, _ in missing_edges_sbs}
        regular  = [(u, v) for u, v, _ in G.edges(data=True) if (u, v) not in miss_set]
        flagged  = [(u, v) for u, v in miss_set if G.has_edge(u, v)]
        if regular:
            nx.draw_networkx_edges(G, pos, edgelist=regular, alpha=0.2,
                edge_color="gray", arrows=True, arrowsize=10, width=1.5,
                connectionstyle="arc3,rad=0.1", ax=ax)
        if flagged:
            nx.draw_networkx_edges(G, pos, edgelist=flagged, alpha=0.6,
                edge_color="#FF6B6B", arrows=True, arrowsize=12, width=3,
                connectionstyle="arc3,rad=0.1", ax=ax)
    else:
        nx.draw_networkx_edges(G, pos, alpha=0.3, edge_color="#4CAF50",
            arrows=True, arrowsize=10, width=2,
            connectionstyle="arc3,rad=0.1", ax=ax)

    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors,
        alpha=0.9, edgecolors="black", linewidths=2, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold", ax=ax)

    stats = f"{G.number_of_nodes()} types · {G.number_of_edges()} edges"
    if highlight_missing:
        stats += f" · {len(missing_edges_sbs)} missing edges (red)"
    ax.set_title(f"{title}\n{stats}", fontsize=14, fontweight="bold", pad=20)
    ax.axis("off")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(28, 12))
_draw_sbs(src_sub, ax1, f"Source Pattern Graph — {TENANT_ALIAS}  (top {TOP_N_SBS})", highlight_missing=True)
_draw_sbs(tgt_sub, ax2, f"Target Pattern Graph  (top {min(TOP_N_SBS, len(top_tgt))})")

from matplotlib.patches import Patch as _P
fig.legend(handles=[
    _P(facecolor="#66b3ff", edgecolor="black", label="Common resource types"),
    _P(facecolor="#ff9999", edgecolor="black", label="Unique to this graph"),
    _P(facecolor="white", edgecolor="#FF6B6B", linewidth=3, label="Missing edges (source → target gap)"),
    _P(facecolor="white", edgecolor="#4CAF50", linewidth=3, label="Captured edges (target)"),
], loc="upper center", ncol=4, fontsize=11, bbox_to_anchor=(0.5, 0.99))
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

print(f"\n📊 Summary:")
print(f"   Common nodes  : {len(common_nodes)}")
print(f"   Missing edges : {len(missing_edges_sbs)}")
if missing_edges_sbs:
    for rel, cnt in Counter(r for _,_,r in missing_edges_sbs).most_common(5):
        print(f"     {rel}: {cnt}")


## Target Pattern Graph Visualization

Same rendering as the source pattern graph above — dashed convex hull boundaries per architectural pattern, thick coloured intra-pattern edges, thin gray cross-pattern edges. Node size ∝ resource count.

In [ ]:
TOP_N_TGT = 80

G_tgt = target_pattern_graph
pattern_matches_tgt = replicator.detected_patterns  # same pattern definitions

# ── Reconstruct edge_counts for target graph ───────────────────────────────────
edge_counts_tgt: dict[tuple, int] = {}
for u, v, data in G_tgt.edges(data=True):
    key = (u, v)
    edge_counts_tgt[key] = edge_counts_tgt.get(key, 0) + data.get("frequency", 1)

# ── Top N connected nodes by degree ───────────────────────────────────────────
tgt_connected = {n for n in G_tgt.nodes() if G_tgt.degree(n) > 0}
tgt_top_names = [
    n for n, _ in sorted(
        ((n, G_tgt.degree(n)) for n in tgt_connected),
        key=lambda x: x[1], reverse=True,
    )[:TOP_N_TGT]
]
G_tgt_f = G_tgt.subgraph(tgt_top_names).copy()

print(f"Target graph    : {G_tgt.number_of_nodes()} nodes, {G_tgt.number_of_edges()} edges")
print(f"Showing top {TOP_N_TGT}  : {G_tgt_f.number_of_nodes()} nodes, {G_tgt_f.number_of_edges()} edges")

# ── Layout ────────────────────────────────────────────────────────────────────
pos_tgt = nx.spring_layout(G_tgt_f, k=3, iterations=50, seed=42)

# ── Assign patterns to nodes ──────────────────────────────────────────────────
node_pm_tgt: dict[str, list] = {}
node_colors_tgt: list = []

for node in G_tgt_f.nodes():
    nps = [p for p, m in pattern_matches_tgt.items() if node in m["matched_resources"]]
    node_pm_tgt[node] = nps
    if nps:
        best = max(nps, key=lambda p: pattern_matches_tgt[p]["completeness"])
        node_colors_tgt.append(list(pattern_matches_tgt.keys()).index(best))
    else:
        node_colors_tgt.append(-1)

node_sizes_tgt = [
    max(200, G_tgt_f.nodes[n].get("count", G_tgt.degree(n)) / 4)
    for n in G_tgt_f.nodes()
]

# ── Separate intra-pattern vs cross-pattern edges ─────────────────────────────
pat_edges_tgt, xpat_edges_tgt = [], []
pat_widths_tgt, xpat_widths_tgt = [], []
pat_colors_tgt = []

for u, v, data in G_tgt_f.edges(data=True):
    freq  = edge_counts_tgt.get((u, v), 1)
    width = max(1, freq / 50)
    shared = set(node_pm_tgt.get(u, [])) & set(node_pm_tgt.get(v, []))
    if shared:
        pat_edges_tgt.append((u, v))
        pat_widths_tgt.append(width * 2.5)
        pat_colors_tgt.append(list(pattern_matches_tgt.keys()).index(list(shared)[0]))
    else:
        xpat_edges_tgt.append((u, v))
        xpat_widths_tgt.append(width * 0.4)

# ── Draw ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(28, 24))
cmap = plt.cm.tab10
legend_tgt = []
hulls_tgt = 0

for pidx, (pname, match) in enumerate(pattern_matches_tgt.items()):
    p_nodes = [n for n in match["matched_resources"] if n in G_tgt_f.nodes()]
    if len(p_nodes) >= 3:
        try:
            pts = np.array([[pos_tgt[n][0], pos_tgt[n][1]] for n in p_nodes])
            center = pts.mean(axis=0)
            pts_exp = center + (pts - center) * 1.15
            hull = ConvexHull(pts_exp)
            color = cmap(pidx / len(pattern_matches_tgt))
            ax.add_patch(Polygon(
                pts_exp[hull.vertices], facecolor=color, alpha=0.08,
                edgecolor=color, linewidth=3, linestyle="--", zorder=1,
            ))
            hulls_tgt += 1
            legend_tgt.append(Patch(facecolor=color, edgecolor=color,
                label=f"{pname} ({match['completeness']:.0f}%)", alpha=0.5))
        except Exception:
            pass

if xpat_edges_tgt:
    nx.draw_networkx_edges(G_tgt_f, pos_tgt, ax=ax,
        edgelist=xpat_edges_tgt, width=xpat_widths_tgt,
        alpha=0.15, edge_color="gray", arrows=True, arrowsize=10,
        arrowstyle="->", connectionstyle="arc3,rad=0.05")

for idx, (u, v) in enumerate(pat_edges_tgt):
    nx.draw_networkx_edges(G_tgt_f, pos_tgt, ax=ax,
        edgelist=[(u, v)], width=pat_widths_tgt[idx], alpha=0.6,
        edge_color=[cmap(pat_colors_tgt[idx] / len(pattern_matches_tgt))],
        arrows=True, arrowsize=15, arrowstyle="->",
        connectionstyle="arc3,rad=0.1")

nx.draw_networkx_nodes(G_tgt_f, pos_tgt, ax=ax,
    node_size=node_sizes_tgt, node_color=node_colors_tgt,
    cmap=cmap, vmin=-1, vmax=len(pattern_matches_tgt) - 1,
    alpha=0.95, edgecolors="black", linewidths=2.5)
nx.draw_networkx_labels(G_tgt_f, pos_tgt, ax=ax, font_size=10, font_weight="bold")

if legend_tgt:
    ax.legend(handles=legend_tgt, loc="upper left", fontsize=10, framealpha=0.95,
              title="Architectural Patterns", title_fontsize=12)

ax.set_title(
    f"Target Pattern Graph — {TENANT_ALIAS}  (top {TOP_N_TGT} nodes by degree)\n"
    "Dashed boundaries = Pattern groupings  ·  Thick coloured edges = Intra-pattern  ·  "
    "Thin gray = Cross-pattern\n"
    "Node colour = Pattern  ·  Node size ∝ resource count",
    fontsize=16, fontweight="bold", pad=25,
)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"\n📊 Target Visualization:")
print(f"   Resource types shown : {len(G_tgt_f.nodes())}")
print(f"   Intra-pattern edges  : {len(pat_edges_tgt)} (coloured)")
print(f"   Cross-pattern edges  : {len(xpat_edges_tgt)} (gray)")
print(f"   Hulls drawn          : {hulls_tgt} / {len(pattern_matches_tgt)} patterns")


## Spectral Distance Evolution

In [ ]:
print("📈 Computing spectral distance evolution …\n")

n_instances   = len(selected_pattern_instances)
n_checkpoints = min(30, n_instances)
checkpoint_indices = sorted(set(
    [0]
    + [round((i + 1) * n_instances / n_checkpoints) - 1 for i in range(n_checkpoints)]
    + [n_instances - 1]
))

evolution_steps, evolution_spectral = [], []
for idx in checkpoint_indices:
    subset = selected_pattern_instances[: idx + 1]
    g = replicator.target_builder.build_from_instances(subset)
    dist = GraphStructureAnalyzer.compute_spectral_distance(source_graph, g)
    evolution_steps.append(idx + 1)
    evolution_spectral.append(dist)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(evolution_steps, evolution_spectral, color='#2E86AB', linewidth=2.5,
        marker='o', markersize=5, alpha=0.85)
ax.fill_between(evolution_steps, evolution_spectral, alpha=0.1, color='#2E86AB')
ax.annotate(f'Final: {evolution_spectral[-1]:.4f}',
            xy=(evolution_steps[-1], evolution_spectral[-1]),
            xytext=(-60, 14), textcoords='offset points',
            fontsize=11, color='#2E86AB', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#2E86AB', lw=1.5))
ax.axhline(y=evolution_spectral[-1], color='#2E86AB', linestyle='--', alpha=0.4, linewidth=1.5)
ax.set_xlabel('Instances Added', fontsize=12, fontweight='bold')
ax.set_ylabel('Spectral Distance  (lower = better)', fontsize=12, fontweight='bold')
ax.set_title(
    f'Spectral Distance Evolution — {TENANT_ALIAS}\n'
    '(approaches source graph topology as more instances are selected)',
    fontsize=13, fontweight='bold', pad=14)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, n_instances + 5)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

mid = len(evolution_steps) // 2
reduction = evolution_spectral[0] - evolution_spectral[-1]
print(f"\n📊 Evolution Summary:")
print(f"   Start  (   1 instance) : {evolution_spectral[0]:.4f}")
print(f"   Mid    ({evolution_steps[mid]:4d} instances) : {evolution_spectral[mid]:.4f}")
print(f"   Final  ({evolution_steps[-1]:4d} instances) : {evolution_spectral[-1]:.4f}")
print(f"   Reduction              : {reduction:.4f}  ({reduction / evolution_spectral[0]:.1%})")

## Replication Fidelity Metrics

Live resource-level comparison between the **source tenant** and the **target subscription** in Neo4j.
Each resource is classified as: exact match, drifted (property differences), missing from target, or missing from source.

In [ ]:
import importlib
import src.validation.resource_fidelity_calculator as _rfc_mod
importlib.reload(_rfc_mod)
from src.validation.resource_fidelity_calculator import ResourceFidelityCalculator, RedactionLevel
from src.utils.session_manager import Neo4jSessionManager
from src.config_manager import Neo4jConfig
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Target subscription (loaded from .env) ────────────────────────────────────
TARGET_SUBSCRIPTION_ID = os.getenv("MULTICLOUD_TARGET_SUBSCRIPTION_ID")
if not TARGET_SUBSCRIPTION_ID:
    raise EnvironmentError("MULTICLOUD_TARGET_SUBSCRIPTION_ID is not set — add it to your .env file")

# ── Connect and compute fidelity ──────────────────────────────────────────────
neo4j_config = Neo4jConfig(uri=NEO4J_URI, user=NEO4J_USER, password=NEO4J_PASSWORD)
session_mgr  = Neo4jSessionManager(neo4j_config)
session_mgr.connect()

calc   = ResourceFidelityCalculator(session_mgr, SUBSCRIPTION_ID, TARGET_SUBSCRIPTION_ID)
result = calc.calculate_fidelity(redaction_level=RedactionLevel.FULL)
m      = result.metrics

PLOT_STATUS_ORDER = ["exact_match", "drifted"]
STATUS_LABELS = {
    "exact_match": "Exact Match",
    "drifted":     "Drifted",
}
STATUS_COLORS = {
    "exact_match": "#2E86AB",
    "drifted":     "#F4A261",
}

counts = {
    "exact_match":    m.exact_match,
    "drifted":        m.drifted,
    "missing_target": m.missing_target,
    "missing_source": m.missing_source,
}
total = m.total_resources
score = m.match_percentage

print(f"Source subscription : {SUBSCRIPTION_ID}")
print(f"Target subscription : {TARGET_SUBSCRIPTION_ID}")
print(f"Total resources     : {total}")
print(f"Fidelity score      : {score:.1f}%  ({counts['exact_match']} exact matches)")
print(f"  Exact Match                  : {counts['exact_match']}")
print(f"  Drifted                      : {counts['drifted']}")
print(f"  Missing (not deployed)       : {counts['missing_target']}")
print(f"  Missing (no source)          : {counts['missing_source']}")
if result.security_warnings:
    print(f"\n[Security] {result.security_warnings[0]}")

# ── Figure 1: Status counts + percentage breakdown ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

ax = axes[0]
x    = np.arange(len(PLOT_STATUS_ORDER))
bars = ax.bar(x, [counts[s] for s in PLOT_STATUS_ORDER],
              color=[STATUS_COLORS[s] for s in PLOT_STATUS_ORDER],
              edgecolor="black", linewidth=1, width=0.45)
for bar, s in zip(bars, PLOT_STATUS_ORDER):
    v = counts[s]
    if v > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                str(v), ha="center", va="bottom", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([STATUS_LABELS[s] for s in PLOT_STATUS_ORDER], fontsize=12)
ax.set_ylabel("Resource Count", fontsize=12)
ax.set_title("Resource Fidelity Status Counts", fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(bottom=0)

ax2 = axes[1]
plot_total = sum(counts[s] for s in PLOT_STATUS_ORDER)
pcts  = [counts[s] / plot_total * 100 if plot_total else 0 for s in PLOT_STATUS_ORDER]
bars2 = ax2.bar([STATUS_LABELS[s] for s in PLOT_STATUS_ORDER], pcts,
                color=[STATUS_COLORS[s] for s in PLOT_STATUS_ORDER],
                edgecolor="black", linewidth=1.2, width=0.45)
for bar, val in zip(bars2, pcts):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
             f"{val:.1f}%", ha="center", va="bottom", fontsize=13, fontweight="bold")
ax2.set_ylabel("% of Deployed Resources", fontsize=12)
ax2.set_title(
    f"Fidelity Distribution — {plot_total} Deployed Resources\n"
    f"Exact-match score: {score:.1f}%",
    fontsize=13, fontweight="bold")
ax2.set_ylim(0, 110)
ax2.tick_params(axis="x", labelsize=12)
ax2.grid(axis="y", alpha=0.3)

plt.suptitle(f"Architecture Replication Fidelity — {TENANT_ALIAS}",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ── Figure 2: Stacked bar by resource type ────────────────────────────────────
import pandas as pd

rows = [
    {
        "short_type": c.resource_type.split("/")[-1].lower(),
        "status":     c.status.value,
    }
    for c in result.classifications
    if c.status.value in PLOT_STATUS_ORDER
]
df = pd.DataFrame(rows)

pivot = (
    df.groupby(["short_type", "status"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=PLOT_STATUS_ORDER, fill_value=0)
)
pivot["total"] = pivot.sum(axis=1)
pivot = pivot.sort_values("total", ascending=False)

all_types = list(pivot.index)
x2        = np.arange(len(all_types))
bottoms   = np.zeros(len(all_types))

fig2, ax3 = plt.subplots(figsize=(14, 6))
for status in PLOT_STATUS_ORDER:
    vals = pivot.get(status, pd.Series(0, index=pivot.index)).values.astype(float)
    ax3.bar(x2, vals, bottom=bottoms, color=STATUS_COLORS[status],
            edgecolor="white", linewidth=0.5, width=0.7)
    for j, (v, b) in enumerate(zip(vals, bottoms)):
        if v >= 2:
            ax3.text(x2[j], b + v / 2, str(int(v)),
                     ha="center", va="center", fontsize=7.5,
                     color="white", fontweight="bold")
    bottoms += vals

ax3.set_xticks(x2)
ax3.set_xticklabels(all_types, rotation=45, ha="right", fontsize=9)
ax3.set_ylabel("Resource Count", fontsize=12)
ax3.set_title(
    f"Fidelity by Resource Type — {TENANT_ALIAS}\n"
    f"Source: {SUBSCRIPTION_ID}  →  Target: {TARGET_SUBSCRIPTION_ID}",
    fontsize=12, fontweight="bold")
ax3.grid(axis="y", alpha=0.3)
ax3.set_ylim(bottom=0)

handles = [mpatches.Patch(color=STATUS_COLORS[s], label=STATUS_LABELS[s])
           for s in PLOT_STATUS_ORDER]
fig2.legend(handles=handles, loc="lower center", ncol=2, fontsize=10,
            bbox_to_anchor=(0.5, -0.05), framealpha=0.9)
plt.tight_layout()
plt.show()

session_mgr.disconnect()

# ── Top drifted properties ────────────────────────────────────────────────────
if m.top_mismatched_properties:
    print("\nTop mismatched properties (drifted resources):")
    for entry in m.top_mismatched_properties[:10]:
        print(f"  {entry['property']:50s}  {entry['count']} resources")

---
# Phase 3: Deployment Complexity Scoring
---

Each selected pattern instance is scored on two dimensions grounded in Microsoft Learn data:

| Dimension | Scale | What it measures |
|---|---|---|
| **Time score** | 1–10 | Provisioning wall-clock duration (gated by the slowest resource) |
| **Complexity score** | 1–10 | Configuration difficulty — prerequisites, IAM surface, networking, quotas |
| **Combined score** | 1–10 | 50 % time + 50 % complexity + coordination penalty for large instances |

**Difficulty bands:** Low < 3.0 · Medium 3–5.5 · High 5.5–7.5 · Critical ≥ 7.5

## Score All Patterns

In [ ]:
import importlib
import src.architectural_pattern_analyzer as _apa_mod
importlib.reload(_apa_mod)
from src.architectural_pattern_analyzer import ArchitecturalPatternAnalyzer

analyzer = ArchitecturalPatternAnalyzer(
    neo4j_uri=NEO4J_URI,
    neo4j_user=NEO4J_USER,
    neo4j_password=NEO4J_PASSWORD,
)

# Build pattern_resources dict from the replicator:
# { pattern_name: [ [resource, ...], ... ] }
pattern_resources_for_scoring = replicator.pattern_resources

print(f"🔢 Scoring deployment complexity for {TENANT_ALIAS} …")
scoring_results = analyzer.score_all_patterns(pattern_resources_for_scoring)

print(f"\n✅ Scored {len(scoring_results['patterns'])} patterns")
print(f"   Overall mean combined score : {scoring_results['overall_mean_combined_score']:.2f} / 10")
print(f"\n{'Pattern':<32} {'Band':<10} {'Combined':>10} {'Time':>8} {'Complexity':>12} {'Instances':>10}")
print("-" * 90)
for pname in scoring_results['ranked']:
    s = scoring_results['patterns'][pname]
    print(
        f"  {pname:<30} {s['difficulty_band']:<10} "
        f"{s['mean_combined_score']:>8.2f}   "
        f"{s['mean_time_score']:>6.2f}   "
        f"{s['mean_complexity_score']:>10.2f}   "
        f"{s['instance_count']:>8}"
    )

## Pattern Difficulty Chart

In [ ]:
BAND_COLORS = {"Low": "#57cc57", "Medium": "#f5a623", "High": "#e05c3a", "Critical": "#8b0000", "Unknown": "#aaaaaa"}

# Only plot patterns that have at least one instance
ranked = [p for p in scoring_results["ranked"] if scoring_results["patterns"][p]["instance_count"] > 0]
patterns_data = scoring_results["patterns"]

labels      = ranked
time_scores = [patterns_data[p]["mean_time_score"]       for p in ranked]
comp_scores = [patterns_data[p]["mean_complexity_score"] for p in ranked]
bands       = [patterns_data[p]["difficulty_band"]       for p in ranked]
bar_colors  = [BAND_COLORS.get(b, "#aaaaaa") for b in bands]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars_t = ax.bar(x - width/2, time_scores, width, label='Time Score',       color='#4A90D9', alpha=0.85, edgecolor='white')
bars_c = ax.bar(x + width/2, comp_scores, width, label='Complexity Score', color='#E8734A', alpha=0.85, edgecolor='white')

for bar, colour in zip(list(bars_t) + list(bars_c), bar_colors * 2):
    bar.set_edgecolor(colour)
    bar.set_linewidth(2.5)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Score (1–10)', fontsize=12, fontweight='bold')
ax.set_ylim(0, 11)
ax.set_title(
    f'Deployment Complexity by Pattern — {TENANT_ALIAS}\n'
    'Bar colour = difficulty band  |  Edge colour = band',
    fontsize=13, fontweight='bold', pad=14
)
ax.axhline(y=scoring_results['overall_mean_combined_score'],
           color='gray', linestyle='--', linewidth=1.5, alpha=0.6)
ax.grid(axis='y', alpha=0.3)

legend_patches = [mpatches.Patch(color=c, label=b) for b, c in BAND_COLORS.items() if b != "Unknown"]
ax.legend(handles=ax.get_legend_handles_labels()[0] + legend_patches,
          labels=ax.get_legend_handles_labels()[1] + [b for b in BAND_COLORS if b != "Unknown"],
          loc='upper right', fontsize=10, ncol=2)

plt.tight_layout()
plt.show()

## Deployment Complexity Heatmap

Shows time score vs complexity score per pattern. Patterns in the top-right are both slow and difficult — prioritise these for IaC automation.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for pname in ranked:  # ranked already excludes zero-instance patterns
    s = patterns_data[pname]
    colour = BAND_COLORS.get(s['difficulty_band'], '#aaaaaa')
    size   = max(60, s['instance_count'] * 3)
    ax.scatter(
        s['mean_time_score'], s['mean_complexity_score'],
        s=size, color=colour, alpha=0.85, edgecolors='black', linewidths=1.2, zorder=3
    )
    ax.annotate(
        pname, (s['mean_time_score'], s['mean_complexity_score']),
        textcoords='offset points', xytext=(6, 4), fontsize=8.5
    )

mid = 5
ax.axvline(x=mid, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.axhline(y=mid, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.fill_betweenx([mid, 10.5], mid, 10.5, alpha=0.04, color='red')
ax.text(9.5, 10.2, 'Hardest', ha='right', color='#cc4444', fontsize=9, style='italic')
ax.text(0.5, 0.5,  'Easiest', ha='left',  color='#336633', fontsize=9, style='italic')

ax.set_xlabel('Time Score  (1=<1 min … 10=45+ min)', fontsize=12, fontweight='bold')
ax.set_ylabel('Complexity Score  (1=trivial … 10=expert)', fontsize=12, fontweight='bold')
ax.set_title(
    f'Time vs Complexity — {TENANT_ALIAS}\n'
    'Bubble size = instance count',
    fontsize=13, fontweight='bold', pad=14
)
ax.set_xlim(0, 11)
ax.set_ylim(0, 11)
ax.grid(True, alpha=0.2)

legend_patches = [mpatches.Patch(color=c, label=b) for b, c in BAND_COLORS.items() if b != "Unknown"]
ax.legend(handles=legend_patches, title='Difficulty Band', loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

## Critical Resource Types

Resource types that most frequently drive high scores (appear as critical resources across instances).

In [ ]:
global_critical: Counter = Counter()
for pname, s in scoring_results['patterns'].items():
    if s['instance_count'] > 0:
        global_critical.update(s['top_critical_resources'])

print(f"🔴 Top critical resource types across all patterns in {TENANT_ALIAS}:\n")
print(f"  {'Resource Type':<35} {'Frequency':<12} {'Time':>6} {'Complexity':>12} {'Time Estimate'}")
print("  " + "-" * 80)
for rtype, freq in global_critical.most_common(15):
    scores = ArchitecturalPatternAnalyzer.RESOURCE_DEPLOYMENT_SCORES.get(
        rtype, ArchitecturalPatternAnalyzer._DEFAULT_DEPLOYMENT_SCORE
    )
    print(
        f"  {rtype:<35} {freq:<12} "
        f"{scores['time_score']:>4}   "
        f"{scores['complexity_score']:>10}   "
        f"{scores['time_estimate']}"
    )

if not global_critical:
    print("  (no critical resources found — all patterns may have been filtered out)")
else:
    fig, ax = plt.subplots(figsize=(12, 5))
    top_items = global_critical.most_common(12)
    rtypes = [r for r, _ in top_items]
    freqs  = [f for _, f in top_items]
    bar_c  = [
        BAND_COLORS.get(
            next(
                (s['difficulty_band'] for s in scoring_results['patterns'].values()
                 if s['instance_count'] > 0 and r in s['top_critical_resources']),
                'Medium'
            ), '#f5a623'
        )
        for r in rtypes
    ]
    ax.barh(rtypes[::-1], freqs[::-1], color=bar_c[::-1], edgecolor='white', alpha=0.85)
    ax.set_xlabel('Number of patterns where this type is critical', fontsize=11, fontweight='bold')
    ax.set_title(f'Most Impactful Resource Types — {TENANT_ALIAS}', fontsize=13, fontweight='bold', pad=14)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## Summary
---

In [ ]:
node_coverage = len(common_nodes) / len(source_nodes_set) * 100 if source_nodes_set else 0
edge_coverage = target_edges / source_edges * 100 if source_edges else 0

if spectral_distance < 0.1:
    quality, quality_icon = "EXCELLENT", "[OK]"
elif spectral_distance < 0.2:
    quality, quality_icon = "GOOD", "[OK]"
elif spectral_distance < 0.3:
    quality, quality_icon = "MODERATE", "[!]"
else:
    quality, quality_icon = "LOW", "[X]"

hardest = scoring_results['ranked'][0]
easiest = scoring_results['ranked'][-1]

print("=" * 72)
print(f"  FULL SUMMARY — {TENANT_ALIAS}")
print("=" * 72)
print(f"  Tenant ID              : {TENANT_ID}")
print(f"  Subscription           : {SUBSCRIPTION_NAME}")
print()
print("  --- Replication Plan ---")
print(f"  Selected instances     : {len(selected_pattern_instances)}")
print(f"  Total resources        : {total_resources}")
print(f"  Node coverage          : {node_coverage:.1f}%")
print(f"  Edge coverage          : {edge_coverage:.1f}%")
print(f"  Spectral distance      : {spectral_distance:.4f}  {quality_icon} {quality}")
print()
print("  --- Deployment Complexity ---")
print(f"  Overall mean score     : {scoring_results['overall_mean_combined_score']:.2f} / 10")
print(f"  Hardest pattern        : {hardest}  "
      f"({scoring_results['patterns'][hardest]['difficulty_band']}, "
      f"score={scoring_results['patterns'][hardest]['mean_combined_score']:.2f})")
print(f"  Easiest pattern        : {easiest}  "
      f"({scoring_results['patterns'][easiest]['difficulty_band']}, "
      f"score={scoring_results['patterns'][easiest]['mean_combined_score']:.2f})")
if global_critical:
    top_crit = global_critical.most_common(3)
    print(f"  Top critical resources : {', '.join(r for r, _ in top_crit)}")
print("=" * 72)